In [1]:
import pandas as pd
df = pd.read_csv('../01-intro/1.9 Bike Sharing/data/day.csv')
print(df.shape)
df.columns

(731, 16)


Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')

In [3]:
y = df['cnt']
X_candidates = df.drop(columns=['cnt', 'casual', 'registered', 'instant'])

print(y.shape)
print(X_candidates.shape)
print(X_candidates.columns.tolist())

(731,)
(731, 12)
['dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']


In [6]:
feature_audit = pd.DataFrame({
    "feature": X_candidates.columns
})

availability_map = {
    "dteday": "known",
    "season": "known",
    "yr": "known",
    "mnth": "known",
    "holiday": "known",
    "weekday": "known",
    "workingday": "known",
    "weathersit": "forecast_needed",
    "temp": "forecast_needed",
    "atemp": "forecast_needed",
    "hum": "forecast_needed",
    "windspeed": "forecast_needed",
}

feature_audit["availability"] = (
    feature_audit["feature"].map(availability_map)
)

print(feature_audit.isnull().sum())
print(feature_audit)

feature         0
availability    0
dtype: int64
       feature     availability
0       dteday            known
1       season            known
2           yr            known
3         mnth            known
4      holiday            known
5      weekday            known
6   workingday            known
7   weathersit  forecast_needed
8         temp  forecast_needed
9        atemp  forecast_needed
10         hum  forecast_needed
11   windspeed  forecast_needed


In [12]:
X_calendar = X_candidates[feature_audit.query("availability == 'known'")["feature"].tolist()]

print(X_calendar.shape)
print(X_calendar.columns.tolist())

forbidden_features = {"casual", "registered", "cnt"}
allowed_features = set(X_calendar.columns)

assert forbidden_features.isdisjoint(allowed_features)

(731, 7)
['dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday']


## Prediction Problem: Five Modeling Assumptions
1. Prediction timing: The model generates a prediction at 6 PM each day for the total number of bike rentals during the following calendar day.
2. Feature Availability: Information that the model can access reliably at prediction time is 'dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday'. 
3. Weather information: We are excluding observed weather variables from our initial model in order to prevent data leakage, since the model will not have the actual weather information during prediction time.
4. Historical data: We are assuming that the 2011-2012 dataset will be relevant to making predictions of future demand. One reason demand patterns from 2011-2012 might not remain reliable in future years is because many factors change as time goes on, so our model will need to keep finding new/updated patterns for future years. Bike-sharing demand could change because of population growth, expansion of bike stations, changes in commuting habits, or changes in transportation infrastructure.
5. Model evaluation: We should divide the historical observations into 70% training data, 15% validation data, and 15% testing data. We should choose a chronological order split strategy because it falls more inline with how we are going to use the model. Once the model is trained and selected, we will keep using it for predicting rental demand for the next day so chronological order suits this scenario better.